# フィルム成長パラメタの最適化

仮想的に（調整する4つの成長パラメータ、目的変数）の値を生成したデータを用いる。
フィルム成長パラメタをベイズ最適化する。

以下はgpt-5の生成コードを元に修正を加えた。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_data(data_name="epitaxy-film-growth"):
    feat_cols = ["Growth Temperature (°C)", "TDMAT Pressure (Torr)", 
                 "N2 Gas Flow (sccm)", "N2 Plasma RF Power (W)"]
    target_col = "log10 Normalized XRD Intensity (Inorm = ITiN(002)/ISTO(002)d)"
    
    df_all = pd.read_csv("../data_calculated/Epitaxial-thin-filmgrowth.csv")

    return df_all, feat_cols, target_col

# epitaxy-filmの成長パラメタと仮想的なXRD intensityを取得する。
g_df_all, g_feat_cols, g_target_col = get_data()

# 目的変数の最大値を求めておく。
g_target_max = g_df_all[g_target_col].max()
print(f"Maximum of '{g_target_col}' = {g_target_max}")

g_df_all

scalerとpcaは文献データの分布から計算しています。
df_allから評価しません。

In [ ]:
import pickle

def load_model(filename = "../data_calculated/Epitaxial-thin-filmgrowth.pkl"):
    """
    保存済みの学習済みパイプラインを読み込み、スケーラとPCAオブジェクトを返す関数。

    Parameters
    ----------
    filename : str, optional
        pickle 形式で保存された学習済みパイプラインのファイルパス。
        デフォルトは "model/best_pipe.pkl"。

    Returns
    -------
    scaler :
        読み込んだパイプライン内の "scaler" ステップ（例: StandardScaler など）。
    pca :
        読み込んだパイプライン内の "pca" ステップ（例: PCA など）。

    Notes
    -----
    本関数は、`sklearn.pipeline.Pipeline` を pickle で保存したファイルを前提としており、
    `named_steps["scaler"]` および `named_steps["pca"]` が存在することを期待している。
    """
    with open(filename, "rb") as f:
        best_pipe = pickle.load(f)
    scaler = best_pipe.named_steps["scaler"]
    pca = best_pipe.named_steps["pca"]
    return scaler,pca

g_X_scaler,g_pca = load_model()
g_X_scaler, g_pca

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

def GP_UCB(df_all, x_scaler, feat_cols, target_col, T = 30, n_init=5, y_thredhold=-4.5, random_seed=10):
    """
    ガウス過程回帰 (Gaussian Process Regression) と UCB (Upper Confidence Bound) に基づく
    ベイズ最適化を、既存データ上で「候補点の中から次に実験すべき条件を選ぶ」形で実行する。

    本関数は以下を行う:
    1. DataFrame から特徴量列と目的変数列を取り出し、X をスケーリング・y を標準化
    2. 目的変数が `y_thredhold` 未満のサンプルから初期点をランダムに `n_init` 個選択
    3. 選択済み点を観測データとして GP を学習し、残りの候補点に対して UCB 取得関数を計算
    4. UCB 値最大の候補点を 1 つ選び、観測集合に追加
    5. 上記の手順を最大 T 回繰り返す
    6. 初期点および BO ループで選ばれた点の履歴を DataFrame として返す

    Parameters
    ----------
    df_all : pandas.DataFrame
        全サンプルを含む DataFrame。`feat_cols` に対応する特徴量列と、
        `target_col` に対応する目的変数列を含んでいる必要がある。
    x_scaler :
        事前に学習済みのスケーラ (例: `sklearn.preprocessing.StandardScaler`)。
        `x_scaler.transform(X_all)` が呼び出せることを前提とする。
    feat_cols : list of str
        特徴量として用いる列名のリスト。ここでは長さ 4 を仮定しており、
        `feat_cols[0]`〜`feat_cols[3]` をそのまま履歴のカラム名として用いる。
    target_col : str
        目的変数（例: 正規化 log10 XRD 強度など）の列名。
    T : int, optional
        ベイズ最適化 (UCB による点選択) を繰り返す最大反復回数。
        デフォルトは 30。
    n_init : int, optional
        初期観測点としてランダムに選ぶサンプル数。
        実際にはデータ数との兼ね合いで `n_init_eff = min(n_init, len(X_all_std))`
        が用いられる。デフォルトは 5。
    y_thredhold : float, optional
        初期点候補を抽出するための閾値。
        `y_all < y_thredhold` を満たすサンプルの中から `n_init` 個を選ぶ。
        デフォルトは -4.5。
    random_seed : int, optional
        乱数シード。初期点のランダム抽出や GP ハイパーパラメータ最適化に用いる。
        デフォルトは 10。

    Returns
    -------
    hist_df : pandas.DataFrame
        ベイズ最適化の履歴をまとめた DataFrame。
        各行は 1 つの選択点（初期点 + BO ループで選ばれた点）に対応し、主な列は以下の通り:
            - "iter" : 反復番号。
              初期点は負の値（`-len(init_idx)+1, ..., 0`）を持ち、
              BO ループで選ばれた点は 1, 2, ..., T の正の値を持つ。
            - "ucb_beta" : その反復における UCB の β_t 値（初期点は NaN）。
            - "chosen_row_in_df_pred" : `df_all` 内での元の行インデックス。
            - 各特徴量列 (feat_cols[0]〜[3]) : 元スケールに逆変換した特徴量値。
            - target_col : 元スケールに逆変換された目的変数値。
            - "ucb_value" : 選択された点における UCB 値。
            - "mu_std_space" : 標準化 y 空間での予測平均。
            - "sigma_std_space" : 標準化 y 空間での予測標準偏差。
            - "kernel_str" : 最適化後カーネルの文字列表現。
            - "const_value", "length_scales", "noise_level" :
              最適化されたカーネルハイパーパラメータ。
    y_scaler : sklearn.preprocessing.StandardScaler
        目的変数 y を標準化するためにフィットされた `StandardScaler` オブジェクト。
        必要に応じて逆変換などに利用できる。

    Notes
    -----
    - カーネルは `ConstantKernel * RBF + WhiteKernel` で構成されており、
      信号の共分散構造 (RBF) と観測ノイズ (WhiteKernel) を同時に学習する。
    - 取得関数は UCB: `mu + sqrt(beta_t) * sigma` を用いており、
      `beta_schedule` で反復 t と次元 d に応じた β_t を計算する。
    - 候補点が尽きた場合は、その時点でループを打ち切る。
    """

    X_all = df_all[feat_cols].values.astype(float)
    y_all = df_all[target_col].values.astype(float)
    
    # ===== X と y の標準化 =====
    X_all_std = x_scaler.transform(X_all)
    
    y_scaler = StandardScaler().fit(y_all.reshape(-1, 1))
    y_all_std = y_scaler.transform(y_all.reshape(-1, 1)).ravel()  # → 1次元に戻す
    
    # ===== GP（RBF） + UCB BO の設定 =====
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(X_all_std.shape[1]),
                                                   length_scale_bounds=(1e-2, 1e3)) \
             + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-9, 1e-2))
    
    def fit_gp(X_obs, y_obs):
        gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False,
                                      n_restarts_optimizer=5, random_state=random_seed)
        gp.fit(X_obs, y_obs)
        return gp
    
    def ucb(mu, std, beta):
        return mu + np.sqrt(beta) * std
    
    def beta_schedule(t, d, delta=0.1):
        return 2.0 * np.log((t**(d/2.0 + 2.0)) * (np.pi**2) / (3.0 * delta))
    
    # ===== 初期点の選択 =====
    rng = np.random.default_rng(random_seed)

    # 見つかりにくくするように
    # y_log_obs < y_thredhold を満たす行のインデックス
    low_idx = np.where(y_all < y_thredhold)[0]
    # 有効な初期点数（データ数を超えないように）
    n_init_eff = min(n_init, len(X_all_std))
    init_idx = rng.choice(low_idx, size=n_init_eff, replace=False)
    print(y_all[init_idx])

    # BOで選択されたデータ
    X_obs = X_all_std[init_idx]
    y_obs = y_all_std[init_idx]

    # これから選択される可能性があるデータ。
    mask = np.ones(len(X_all_std), dtype=bool)
    mask[init_idx] = False
    X_cand = X_all_std[mask]
    y_cand = y_all_std[mask]
    orig_idx_cand = np.arange(len(X_all_std))[mask]  
    
    
    history = []

    # ===== 初期選択値を history に追加 =====
    # ここでは UCB などはまだ定義されていないので NaN を入れておく
    X_init_orig = X_all[init_idx]      # 元スケールの X
    y_init_orig = y_all[init_idx]      # 元スケールの y

    for i, idx in enumerate(init_idx):
        x_orig = X_init_orig[i]
        y_orig = float(y_init_orig[i])

        history.append({
            "iter": -len(init_idx)+i+1,  # 初期点のインデックス はマイナス
            "ucb_beta": np.nan,
            "chosen_row_in_df_pred": int(idx),
            feat_cols[0]: x_orig[0],
            feat_cols[1]: x_orig[1],
            feat_cols[2]: x_orig[2],
            feat_cols[3]: x_orig[3],
            target_col: y_orig,
            "ucb_value": np.nan,
            "mu_std_space": np.nan,
            "sigma_std_space": np.nan,
            "kernel_str": None,
            "const_value": np.nan,
            "length_scales": np.nan,
            "noise_level": np.nan})

    # ===== BOループ =====

    for t in range(1, T + 1):
        gp = fit_gp(X_obs, y_obs)

        # カーネルハイパーパラメータを抽出
        kernel_opt = gp.kernel_
        kernel_params = kernel_opt.get_params()
        
        mu, std = gp.predict(X_cand, return_std=True)
        beta_t = beta_schedule(t + n_init, d=X_all_std.shape[1], delta=0.1)
        acq = ucb(mu, std, beta=beta_t)
        best_idx = np.argmax(acq)
    
        # 新しい点
        x_new = X_cand[best_idx].reshape(1, -1)
        y_new = y_cand[best_idx].reshape(1, )
    
        # 逆変換して人間可読な値に戻す
        x_new_orig = x_scaler.inverse_transform(x_new)[0]
        y_new_orig = y_scaler.inverse_transform(y_new.reshape(-1,1)).ravel()[0]
    
        history.append({
            "iter": t,
            "ucb_beta": beta_t,
            "chosen_row_in_df_pred": int(orig_idx_cand[best_idx]),
            feat_cols[0]: x_new_orig[0],
            feat_cols[1]: x_new_orig[1],
            feat_cols[2]: x_new_orig[2],
            feat_cols[3]: x_new_orig[3],
            target_col: float(y_new_orig),
            "ucb_value": float(acq[best_idx]),
            "mu_std_space": float(mu[best_idx]),
            "sigma_std_space": float(std[best_idx]),
            # --- ここから kernel の最適パラメータ ---
            "kernel_str": str(kernel_opt),
            "const_value": kernel_params['k1__k1__constant_value'],
            "length_scales": kernel_params['k1__k2__length_scale'],
            "noise_level": kernel_params['k2__noise_level']            
        })
    
        # 観測点更新
        X_obs = np.vstack([X_obs, x_new])
        y_obs = np.concatenate([y_obs, y_new])
    
        # 候補点から削除
        X_cand = np.delete(X_cand, best_idx, axis=0)
        y_cand = np.delete(y_cand, best_idx, axis=0)
        orig_idx_cand = np.delete(orig_idx_cand, best_idx, axis=0)
    
        if len(X_cand) == 0:
            print("候補点がなくなったため終了")
            break
    
    # ===== 結果をDataFrameにまとめ =====
    
    
    hist_df = pd.DataFrame(history)
    print(hist_df.head())

    return hist_df, y_scaler

g_hist_df, g_y_scaler = GP_UCB(g_df_all, g_X_scaler, g_feat_cols, g_target_col)

### prompt
gpt-5
```prompt
ベイズ最適化でWhiteKernelを入れる目的は？
```

### answer
1. 観測値に含まれるノイズをきちんとモデル化するため
1. カーネル行列を数値的に安定させるため（$K + \sigma_n^2 I$)
1. ノイズレベルをデータから学習するため
1. ベイズ最適化で、過信しない・現実的な不確実性推定をするため

補足：
ベイズで用いるGaussian Processの学習は訓練データ・テストデータという分割が無いので、過学習しがちなためノイズを含める。


In [ ]:
g_hist_df["kernel_str"].values

In [ ]:
g_hist_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_GP_path(df_all, hist_df, scaler, pca, feat_cols, target_col):
    """
    PCA 2 次元空間上に全データをプロットし、その上に
    ベイズ最適化 (GP + UCB) で選択された点の「探索パス」を重ね描きする。

    本関数は以下の処理を行う:
    1. `df_all` から特徴量列 `feat_cols` と目的変数 `target_col` を取り出す。
    2. 特徴量を `scaler` でスケーリングし、`pca` によって 2 次元 (PC1, PC2) に射影する。
       その結果を背景点として散布図に描画し、`target_col` の値で色付けする。
    3. `hist_df` からベイズ最適化で選択された条件（`feat_cols`）と反復番号 `iter` を取り出し、
       同様に PCA 空間に射影して、赤い点と番号で描画する。
    4. 選択された点同士を矢印で結び、探索の順番（パス）を可視化する。

    Parameters
    ----------
    df_all : pandas.DataFrame
        全サンプルを含む DataFrame。
        少なくとも `feat_cols` に対応する特徴量列と `target_col` を含むこと。
    hist_df : pandas.DataFrame
        ベイズ最適化の履歴を格納した DataFrame。
        少なくとも以下の列を含んでいることを前提とする:
          - 各特徴量列 `feat_cols[i]`（元スケールの値）
          - "iter" : その点が選択された反復番号
    scaler :
        学習済みスケーラオブジェクト。
        `scaler.transform(X)` が呼び出せることを前提とする（例: StandardScaler）。
    pca :
        学習済み PCA オブジェクト。
        `pca.transform(X_scaled)` が呼び出せ、少なくとも 2 主成分を持つことを前提とする。
    feat_cols : list of str
        特徴量として用いる列名のリスト。
        `df_all` および `hist_df` の両方に同じ列名が存在している必要がある。
    target_col : str
        目的変数の列名。
        背景散布図の色付けに使用される。

    Notes
    -----
    - 背景の散布図は `df_all` の全サンプルを PCA(2D) 空間に射影し、
      `target_col` の値に応じて "viridis" カラーマップで色付けしている。
    - ベイズ最適化で選択された点は赤い点と反復番号で表示され、
      矢印により選択順（探索パス）を示す。
    - 図は `plt.show()` によってその場で表示され、関数の戻り値はない。
    """

    # ---- 1) 全データを PCA(2D) に射影、色＝y_all ----
    X_all = df_all[feat_cols].values 
    y_all = df_all[target_col].values
    
    X_all_pca = pca.transform(scaler.transform(X_all))
    
    plt.figure(figsize=(9,7))
    bg = plt.scatter(
        X_all_pca[:, 0], X_all_pca[:, 1],
        c=y_all, cmap="viridis", s=60, edgecolors="k", alpha=0.2
    )
    cb = plt.colorbar(bg); cb.set_label(target_col)
    
    # ---- 2) ベイズ最適化で選んだ点を PCA(2D) に射影して重ね描き ----
    # hist_df には "X_temp","X_press","X_flow","X_rf","iter" が入っている前提
    X_hist_orig = hist_df[feat_cols].values
    iters = hist_df["iter"].values
    
    X_hist_pca = pca.transform(scaler.transform(X_hist_orig))
    
    Xp = X_hist_pca
    its = iters
    
    # 探索点を番号付きで描画 & 矢印で結ぶ
    for i in range(len(Xp)-1):
        x0, y0 = Xp[i]
        x1, y1 = Xp[i+1]
        # 点 + 番号
        plt.scatter(x0, y0, c="red", s=100, alpha = 0.2, edgecolors="white", zorder=3)
        plt.annotate(str(its[i]), (x0, y0), textcoords="offset points",
                     xytext=(6, 4), fontsize=14, color="black", weight="bold")
        # 矢印（i -> i+1）
        plt.annotate("",
                     xy=(x1, y1), xytext=(x0, y0),
                     arrowprops=dict(arrowstyle="->", color="red", lw=2, alpha=0.3))
    
    # 終点（最後の番号）も描く
    x_last, y_last = Xp[-1]
    plt.scatter(x_last, y_last, c="red", s=100, alpha = 0.2, edgecolors="white", zorder=3)
    plt.annotate(str(its[-1]), (x_last, y_last), textcoords="offset points",
                 xytext=(6, 4), fontsize=14, color="black", weight="bold")
    
    # ---- 軸やタイトル ----
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    plt.title("PCA(2D) of all X (colored by target) with BO path overlay")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_GP_path(g_df_all, g_hist_df, g_X_scaler, g_pca, g_feat_cols, g_target_col)


In [ ]:
import matplotlib.pyplot as plt

def plot_GP_history(hist_df, y_scaler, target_col, target_max):
    """
    ベイズ最適化の履歴を、各種指標との関係として可視化する。

    以下の 4 つのプロットを 2×2 のサブプロットで描画する:

    1. Iteration vs 目的変数 (hist_df[target_col])
       - 各反復で選択された条件における目的変数値をプロットし、
         目標値 `target_max` を横線として表示する。
    2. Iteration vs sigma_std_space
       - ガウス過程の予測標準偏差（標準化 y 空間）と反復番号の関係をプロットする。
    3. Iteration vs μ
       - 標準化 y 空間での予測平均 `mu_std_space` を `y_scaler` で元スケールに戻した値と、
         反復番号との関係をプロットする。
    4. y_log_obs vs μ
       - x 軸に `hist_df[target_col]`（観測 or 元スケールの値）、
         y 軸に (3) で復元した μ をとった散布図を描画し、
         点の色とサイズで反復番号を表現する。
       - 対角線 y = x を描画して、予測と観測の一致具合を視覚的に確認できるようにする。
       - x = `target_max` に縦線を描画する。

    Parameters
    ----------
    hist_df : pandas.DataFrame
        ベイズ最適化の履歴を格納した DataFrame。
        少なくとも以下の列を含むことを前提とする:
            - "iter" : 反復番号
            - target_col : 目的変数（元スケール）の値
            - "mu_std_space" : 標準化 y 空間での GP 予測平均
            - "sigma_std_space" : 標準化 y 空間での GP 予測標準偏差
    y_scaler : sklearn.preprocessing.StandardScaler
        目的変数 y の標準化に用いた `StandardScaler`。
        `mu_std_space` を元スケールに戻すのに使用する。
    target_col : str
        目的変数の列名（例: "y_log_obs" など）。
    target_max : float
        目標となる目的変数の値。
        1 枚目のプロットで横線として、4 枚目のプロットでは縦線として描画される。

    Notes
    -----
    - μ の逆変換は `y_scaler.inverse_transform` を用いて行う。
    - 4 枚目の散布図では、`hist_df["iter"]` に応じて点の色とサイズを変え、
      どの反復で得られた点かが視覚的に分かるようにしている。
    """

    plt.figure(figsize=(12, 8))
    
    # ---- 1. iter vs y_log_obs ----
    plt.subplot(2, 2, 1)
    plt.plot(hist_df["iter"], hist_df[target_col], marker="o", color="blue")
    plt.xlabel("Iteration")
    plt.ylabel("Observed log10(Inorm)")
    plt.title("Iteration vs y_log_obs")
    plt.axhline(y=g_target_max)
    plt.grid(True)
    
    # ---- 2. iter vs ucb_value ----
    plt.subplot(2, 2, 2)
    plt.plot(hist_df["iter"], hist_df["sigma_std_space"], marker="o", color="red")
    plt.xlabel("Iteration")
    plt.ylabel("sigma_std_space")
    plt.title("Iteration vs sigma_std_space")
    plt.grid(True)
    
    # ---- 3. iter vs mu_std_space ----
    # muはy_scalerを用いた。
    mu = y_scaler.inverse_transform(hist_df["mu_std_space"].values.reshape(-1,1))
    
    plt.subplot(2, 2, 3)
    plt.plot(hist_df["iter"], mu, marker="o", color="green")
    plt.xlabel("Iteration")
    plt.ylabel("μ")
    plt.title("Iteration vs μ")
    plt.grid(True)
    
    # ---- 4. y_log_obs vs mu ----
    plt.subplot(2, 2, 4)
    plt.scatter(hist_df[target_col], mu, 
                c=hist_df["iter"], cmap="viridis", s=hist_df["iter"]*5, edgecolors="k")
    plt.colorbar(label="Iteration")
    plt.xlabel("Observed log10(Inorm)")
    plt.ylabel("μ")
    plt.title("sized and colored by iteration")
    plt.axvline(x=g_target_max)
    
    # === 対角線を追加 ===
    lims = [
        min(plt.xlim()[0], plt.ylim()[0]),
        max(plt.xlim()[1], plt.ylim()[1])
    ]
    plt.plot(lims, lims, 'r--', lw=2, label="y = x")
    
    plt.xlim(lims)
    plt.ylim(lims)
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_GP_history(g_hist_df, g_y_scaler,g_target_col, g_target_max)

'Observed log10(Inorm)'にある直線は最大値を表す。
